# 14시 기온·습도 예측 — 학습 노트북

제1회 시스템경영공학과 데이터분석 경진대회 · 제출물 ②

**과제**: GK2A 정지위성 관측만으로 전국 96개 지점의 14시 기온(TA)·습도(HM)를
맞춘다. 점수는 `RMSE_TA + 0.1 x RMSE_HM` 이라 기온이 약 2/3 를 차지한다.

**제약**: ASOS 지상관측은 *학습 라벨로만* 쓸 수 있고 입력으로는 못 쓴다
(규칙 2.2/2.3). 그래서 추론 시점에 우리가 아는 것은 위성 영상, 지점 좌표·고도,
날짜뿐이다.

**인과성**: 대상은 14시(KST) = 05 UTC. 그 시각까지 관측된 영상만 입력이다.
당일 00·02·04·05 UTC(KST 09·11·13·14시)와 전날 18·21 UTC(KST 새벽 3·6시)를 쓴다.

이 노트북을 위에서 아래로 실행하면 제출한 `model.pkl` 이 그대로 나온다
(시드 고정 + LightGBM `deterministic=True`).

## 1. 왜 이 구조인가 — 타깃을 먼저 뜯어봤다

14시 기온의 분산을 세 성분으로 나눠 봤다.

| 성분 | 분산 | 비중 | 무엇으로 잡히나 |
|---|---:|---:|---|
| 날짜간 (그날의 기단) | 9.31 | 66% | 전국 편차·이웃 지점 피처 |
| 지점간 (고정 특성) | 1.31 | 9% | 위경도·고도·지점번호 |
| 날짜x지점 상호작용 | 3.61 | 25% | **여기가 진짜 어려운 부분** |

즉 성능의 3분의 2는 "오늘이 더운 날인가"를 맞히는 문제고, 그건 한 지점의
화소값이 아니라 **전국 패턴**에 들어있다. 이 관찰이 아래 피처 설계를 결정했다.

남은 상호작용 성분을 실측 ASOS 변수와 대조해 보면:

    습도 -0.476 · 일사 +0.383 · 지면온도 +0.373 · 전운량 -0.258 · 풍속 +0.053

습도·일사·지면온도가 지배적인데 셋 다 입력 금지다. 그래서 각각의 위성 대응물
(수증기 흡수차, 반사도, 적외 최저휘도)을 피처로 만들어 대신 쓴다.

## 2. 물리 배경

**분리대기창(split-window)**. KMA 공식 GK2A 지표면온도 알고리즘(ATBD)은

    LST = A0 + A1*T13 + A2*(T13 - T15)

를 쓴다. IR105(10.35um)와 IR123(12.37um)은 파장이 가까워 지표 신호는 거의
같지만 **수증기 흡수량이 다르다**. 그래서 두 채널의 차이가 대기 수증기 효과를
지워주는 보정항이 된다. 보조변수로 위성 천정각이 들어가는데 경로 길이 효과라
`sec(theta)-1` 형태다.

**GK2A DN 은 밝기온도와 반대**. 실측 상관이 전부 음수(-0.46~-0.56)였다.
따라서 창 안의 **최솟값(min9)이 물리적으로 가장 따뜻한 화소** = 구름에 덜 가린
맑은 하늘 대리값이다. 이 부호를 잘못 잡으면 피처 의미가 뒤집힌다.

## 3. 모듈 정의

재현을 위해 사용한 코드를 노트북 안에 그대로 써 넣는다.

In [ ]:
import pathlib
for d in ("sme", "sme/core", "sme/model"):
    pathlib.Path(d).mkdir(exist_ok=True)
    (pathlib.Path(d) / "__init__.py").touch()
print("패키지 준비 완료")

In [ ]:
%%writefile sme/core/paths.py
"""저장소 안의 표준 경로.

스크립트가 어느 패키지 깊이에 있든 같은 곳을 가리키도록 저장소 루트를
`__file__` 에서 거슬러 올라가 계산한다. `data/` 아래를 가리키는 상대경로
문자열은 저장소 루트에서 실행하는 것을 전제로 한다.
"""
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]

DATA = ROOT / "data"                 # 수집한 위성·ASOS 자료
REFERENCE = DATA / "reference"       # 지점 목록 등 손으로 관리하는 표
OUTPUTS = ROOT / "outputs"           # 제출 파일 등 산출물
API_KEY_FILE = ROOT / "apikey.txt"   # 커밋 대상 아님. README 의 API 키 설정 참고
MODEL_FILE = ROOT / "model.pkl"      # 학습 결과. 크기 때문에 커밋하지 않는다


In [ ]:
%%writefile sme/core/grid.py
"""GK2A 격자 <-> 위경도 변환, 지점 주변 화소 추출.

GK2A KO 영역은 Lambert Conformal Conic 투영이고, 투영 파라미터가 NetCDF
전역속성에 전부 들어있다. 그래서 파일에서 직접 읽어 변환기를 만들면
채널 해상도(0.5/1/2km)가 달라도 같은 코드로 처리된다.
"""
from __future__ import annotations

from functools import lru_cache

import numpy as np
import pandas as pd
from pyproj import CRS, Transformer


@lru_cache(maxsize=8)
def _transformer(lat0: float, lon0: float, sp1: float, sp2: float) -> Transformer:
    crs = CRS.from_proj4(
        f"+proj=lcc +lat_1={sp1} +lat_2={sp2} +lat_0={lat0} +lon_0={lon0} "
        "+x_0=0 +y_0=0 +a=6378137 +rf=298.257222101 +units=m +no_defs"
    )
    return Transformer.from_crs("EPSG:4326", crs, always_xy=True)


def lonlat_to_rowcol(lon, lat, meta: dict) -> tuple[np.ndarray, np.ndarray]:
    """위경도 -> (row, col). meta 는 read_gk2a 가 돌려준 전역속성 dict."""
    tf = _transformer(
        float(meta["origin_latitude"]),
        float(meta["central_meridian"]),
        float(meta["standard_parallel1"]),
        float(meta["standard_parallel2"]),
    )
    x, y = tf.transform(np.asarray(lon, float), np.asarray(lat, float))
    ps = float(meta["pixel_size"])
    col = (x - float(meta["upper_left_easting"])) / ps
    row = (float(meta["upper_left_northing"]) - y) / ps
    return row, col


def extract_windows(img: np.ndarray, rows, cols, half: int = 4) -> np.ndarray:
    """각 지점 주변 (2*half+1)^2 창을 잘라 (n_points, w, w) 로 돌려준다.

    격자 밖이거나 창이 잘리는 지점은 NaN 으로 채운다.
    half=4 면 9x9 (2km 채널 기준 18km 반경).
    """
    w = 2 * half + 1
    n = len(rows)
    out = np.full((n, w, w), np.nan, dtype=np.float32)
    H, W = img.shape
    r0 = np.rint(np.asarray(rows)).astype(int)
    c0 = np.rint(np.asarray(cols)).astype(int)

    for i in range(n):
        rs, re = r0[i] - half, r0[i] + half + 1
        cs, ce = c0[i] - half, c0[i] + half + 1
        if rs < 0 or cs < 0 or re > H or ce > W:
            continue  # 가장자리 지점은 NaN 유지
        out[i] = img[rs:re, cs:ce].astype(np.float32)
    return out


def window_features(win: np.ndarray, prefix: str) -> dict[str, np.ndarray]:
    """창 배열 (n, w, w) -> 지점별 요약 통계.

    중심 화소만 쓰면 구름 가장자리에서 값이 널뛴다. 주변 통계를 같이 넣으면
    그 노이즈가 줄고, 표준편차 자체가 '구름이 깨져 있는지'를 알려준다.

    방향성도 함께 뽑는다. 평균 하나로 뭉개면 '서쪽은 구름, 동쪽은 맑음' 같은
    공간 배치가 사라지는데, 동해안의 큰 오차(푄·해풍)가 바로 그런 배치에서
    온다. 사분면 평균과 동서/남북 기울기가 그 정보를 담는다.
    """
    n, w, _ = win.shape
    c = w // 2
    flat = win.reshape(n, -1)
    inner = win[:, c - 1:c + 2, c - 1:c + 2].reshape(n, -1)
    with np.errstate(all="ignore"):
        # 배열은 [row, col] = [북->남, 서->동] 순서다
        north = np.nanmean(win[:, :c, :].reshape(n, -1), axis=1)
        south = np.nanmean(win[:, c + 1:, :].reshape(n, -1), axis=1)
        west = np.nanmean(win[:, :, :c].reshape(n, -1), axis=1)
        east = np.nanmean(win[:, :, c + 1:].reshape(n, -1), axis=1)
        return {
            f"{prefix}_c":    win[:, c, c],
            f"{prefix}_m3":   np.nanmean(inner, axis=1),
            f"{prefix}_m9":   np.nanmean(flat, axis=1),
            f"{prefix}_sd9":  np.nanstd(flat, axis=1),
            f"{prefix}_min9": np.nanmin(flat, axis=1),
            f"{prefix}_max9": np.nanmax(flat, axis=1),
            # 공간 배치: 기울기가 이류 방향과 지형 대비를 담는다
            f"{prefix}_gEW":  east - west,
            f"{prefix}_gNS":  south - north,
            f"{prefix}_gW":   west,
            f"{prefix}_gE":   east,
        }


# ---------------------------------------------------------------- 기하

# GK2A 는 정지궤도라 '통과 시각' 이 없다. 동경 128.2도 상공에 고정되어
# 같은 반구를 계속 관측한다. 그래서 위성 기하는 지점마다 상수다.
GK2A_SUB_LON = 128.2
GEO_ALT_KM = 35786.0
EARTH_R_KM = 6378.137


def satellite_zenith(lon, lat, sub_lon: float = GK2A_SUB_LON) -> np.ndarray:
    """지점에서 본 위성의 천정각(도).

    비스듬히 볼수록 복사가 통과하는 대기 경로가 길어져 관측값이 달라진다.
    정지위성이라 지점당 상수 -> 한 번 계산해 피처로 붙이면 된다.
    """
    lon = np.radians(np.asarray(lon, float) - sub_lon)
    lat = np.radians(np.asarray(lat, float))
    # 지심각
    psi = np.arccos(np.clip(np.cos(lat) * np.cos(lon), -1, 1))
    r = EARTH_R_KM + GEO_ALT_KM
    # 삼각형(지구중심-지점-위성)에서 천정각
    return np.degrees(np.arctan2(r * np.sin(psi),
                                 r * np.cos(psi) - EARTH_R_KM))


def solar_position(when, lon, lat) -> tuple[np.ndarray, np.ndarray]:
    """태양 천정각·방위각(도). when 은 UTC 시각(스칼라 또는 배열).

    NOAA 근사식. 기온 일변화의 원인이자 가시광 채널 밝기의 기준이라
    시각을 그냥 숫자로 넣는 것보다 훨씬 물리적인 피처가 된다.
    """
    t = pd.to_datetime(when, utc=True)
    t = pd.DatetimeIndex(np.atleast_1d(t))
    lon = np.asarray(lon, float)
    lat = np.asarray(lat, float)

    # 율리우스 세기
    jd = t.to_julian_date().to_numpy()
    jc = (jd - 2451545.0) / 36525.0

    geom_mean_lon = (280.46646 + jc * (36000.76983 + jc * 0.0003032)) % 360
    geom_mean_anom = 357.52911 + jc * (35999.05029 - 0.0001537 * jc)
    ecc = 0.016708634 - jc * (0.000042037 + 0.0000001267 * jc)
    m = np.radians(geom_mean_anom)
    sun_eq = (np.sin(m) * (1.914602 - jc * (0.004817 + 0.000014 * jc))
              + np.sin(2 * m) * (0.019993 - 0.000101 * jc)
              + np.sin(3 * m) * 0.000289)
    true_lon = geom_mean_lon + sun_eq
    omega = 125.04 - 1934.136 * jc
    app_lon = np.radians(true_lon - 0.00569 - 0.00478 * np.sin(np.radians(omega)))

    e0 = (23 + (26 + (21.448 - jc * (46.815 + jc * (0.00059 - jc * 0.001813)))
                / 60) / 60)
    oblique = np.radians(e0 + 0.00256 * np.cos(np.radians(omega)))
    decl = np.arcsin(np.sin(oblique) * np.sin(app_lon))

    # 균시차 (분)
    y = np.tan(oblique / 2) ** 2
    gml = np.radians(geom_mean_lon)
    eot = 4 * np.degrees(
        y * np.sin(2 * gml) - 2 * ecc * np.sin(m)
        + 4 * ecc * y * np.sin(m) * np.cos(2 * gml)
        - 0.5 * y * y * np.sin(4 * gml) - 1.25 * ecc * ecc * np.sin(2 * m)
    )

    minutes = (t.hour * 60 + t.minute + t.second / 60).to_numpy()
    true_solar = (minutes + eot + 4 * lon) % 1440
    hour_angle = np.radians(np.where(true_solar / 4 < 0,
                                     true_solar / 4 + 180,
                                     true_solar / 4 - 180))

    latr = np.radians(lat)
    cos_z = (np.sin(latr) * np.sin(decl)
             + np.cos(latr) * np.cos(decl) * np.cos(hour_angle))
    zenith = np.degrees(np.arccos(np.clip(cos_z, -1, 1)))

    az = np.degrees(np.arctan2(
        np.sin(hour_angle),
        np.cos(hour_angle) * np.sin(latr) - np.tan(decl) * np.cos(latr),
    )) + 180
    return zenith, az % 360


In [ ]:
%%writefile sme/core/evaluation.py
"""확정된 규칙에 맞춘 평가.

  타깃  : 매일 14:00 KST 의 TA / HM
  지표  : Score = RMSE_TA + 0.1 * RMSE_HM   (기온이 지배)
  제약  : 대상 시각(14:00 KST = 05:00 UTC)'까지' 관측된 위성만 입력 가능

인과성이 핵심이다. 05 UTC 이후 영상은 정답 시각 이후를 보는 것이라
쓰면 규정 위반이고, 검증 성능도 부풀려진다. 그래서 피처를 만들 때
시각을 UTC 로 환산해 05시 이하만 남긴다.
"""
from __future__ import annotations

import glob

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

SEED = 42
TARGET_HOUR_KST = 14
TARGET_HOUR_UTC = TARGET_HOUR_KST - 9  # 05 UTC
WIN_STATS = ("c", "m9", "sd9", "min9", "max9")

# 당일 받는 시각(UTC). 05 = 14 KST 로 정답과 같은 시각이라 가장 중요하다.
HOURS_CUR = (0, 2, 4, 5)
# 전날 늦은 시각. 오늘 05 UTC 보다 이르므로 규정상 사용 가능하며,
# 밤사이 변화를 담는다.
HOURS_PREV = (18, 21)

PARAMS = dict(objective="regression", metric="rmse", learning_rate=0.05,
              num_leaves=15, min_data_in_leaf=40, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1, verbose=-1,
              seed=SEED, deterministic=True, force_row_wise=True)


def score(rmse_ta: float, rmse_hm: float) -> float:
    return rmse_ta + 0.1 * rmse_hm


def load_labels() -> pd.DataFrame:
    """14:00 KST 관측값. 결측(-99/-9)은 이미 NaN 이며 채점에서도 제외된다."""
    # 전 시각 파일(2023~2025) + 14시만 받은 파일(2019~2022)을 함께 읽는다
    files = sorted(glob.glob("data/asos_20??0601_*.parquet")
                   + glob.glob("data/asos14_*.parquet"))
    d = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    d["TM"] = pd.to_datetime(d["TM"])
    d = d[d["TM"].dt.hour == TARGET_HOUR_KST].copy()
    d = d.drop_duplicates(subset=["STN", "TM"])
    d["date"] = d["TM"].dt.normalize()
    return d[["STN", "date", "TA", "HM"]].rename(
        columns={"TA": "TA14", "HM": "HM14"})


def build_climatology(lab: pd.DataFrame, window: int = 15) -> pd.DataFrame:
    d = lab.copy()
    d["DOY"] = d["date"].dt.dayofyear
    recs = []
    for doy in sorted(d["DOY"].unique()):
        gap = np.abs(d["DOY"] - doy)
        sel = d[np.minimum(gap, 365 - gap) <= window]
        g = sel.groupby("STN")[["TA14", "HM14"]].mean()
        g.columns = ["TA_CLIMO", "HM_CLIMO"]
        recs.append(g.reset_index().assign(DOY=doy))
    return pd.concat(recs, ignore_index=True)


def causal_satellite(channels: list[str]) -> pd.DataFrame:
    """대상일 05 UTC 이하의 위성만 모아 일 단위로 접는다.

    05 UTC 초과분은 정답 시각 이후라 버린다. 전날 늦은 시각은 오늘 05 UTC
    이전이므로 합법이고, 하루 사이의 변화(경향)를 담을 수 있어 유용하다.
    """
    files = sorted(glob.glob("data/sat_final/*.parquet"))
    sat = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    sat["utc_hour"] = sat["time_utc"].dt.hour
    sat["date"] = sat["time_utc"].dt.normalize()

    cols = [f"{ch}_{s}" for ch in channels for s in WIN_STATS
            if f"{ch}_{s}" in sat.columns]

    # 시각별로 나눠 붙인다. 평균으로 뭉개면 정답 시각(05 UTC)의 정보가
    # 이른 아침 관측에 희석된다 — 실제로 나눠 넣는 쪽이 확실히 낫다.
    cur = sat[sat["utc_hour"] <= TARGET_HOUR_UTC].pivot_table(
        index=["STN", "date"], columns="utc_hour", values=cols)
    cur.columns = [f"{a}_h{b:02d}" for a, b in cur.columns]

    # 전날 06 UTC 이후 — 오늘 05 UTC 보다 이르므로 사용 가능
    prev = sat[sat["utc_hour"] > TARGET_HOUR_UTC].copy()
    prev["date"] = prev["date"] + pd.Timedelta(days=1)
    prv = prev.pivot_table(index=["STN", "date"], columns="utc_hour",
                           values=cols)
    prv.columns = [f"{a}_p{b:02d}" for a, b in prv.columns]

    out = cur.join(prv, how="outer")

    # 분리대기창 보정항 — KMA 공식 LST 알고리즘의 (T13 - T15) 에 해당한다.
    # 두 창채널은 지표 신호가 거의 같고 수증기 흡수만 다르므로, 차이가
    # 대기 효과를 상쇄한다. 트리는 이 선형 조합을 스스로 못 만들기 때문에
    # 명시적으로 넣어준다.
    for tag in [f"h{h:02d}" for h in HOURS_CUR] + [f"p{h:02d}" for h in HOURS_PREV]:
        for st in ("c", "m9"):
            a, b = f"IR105_{st}_{tag}", f"IR123_{st}_{tag}"
            if a in out.columns and b in out.columns:
                out[f"SWD_{st}_{tag}"] = out[a] - out[b]

        # 청천 화소 기준 분리대기창.
        # GK2A DN 은 온도와 반대 방향이라(상관 -0.46~-0.56) min9 = DN 최솟값이
        # 물리적으로 가장 뜨거운 화소, 즉 청천 지표에 해당한다. KMA 공식
        # 알고리즘도 청천 육지 화소에서만 지표면온도를 산출한다.
        a, b = f"IR105_min9_{tag}", f"IR123_min9_{tag}"
        if a in out.columns and b in out.columns:
            out[f"SWDmin_{tag}"] = out[a] - out[b]
        # 창 안의 최대-최소 = 구름과 청천이 얼마나 섞였는지 (구름량 대리)
        a, b = f"IR105_max9_{tag}", f"IR105_min9_{tag}"
        if a in out.columns and b in out.columns:
            out[f"CLD_{tag}"] = out[a] - out[b]

    for col in cols:
        # 직전 1시간 변화: 구름이 막 끼거나 걷히는 중인지
        if f"{col}_h05" in out.columns and f"{col}_h04" in out.columns:
            out[f"{col}_d1h"] = out[f"{col}_h05"] - out[f"{col}_h04"]
        # 밤사이 변화: 기단 교체나 구름 유입
        if f"{col}_h05" in out.columns and f"{col}_p21" in out.columns:
            out[f"{col}_dnt"] = out[f"{col}_h05"] - out[f"{col}_p21"]
    return out.reset_index()


def assemble(channels: list[str]) -> pd.DataFrame:
    lab = load_labels()
    climo = build_climatology(lab)   # 비교 평가용일 뿐, 모델 입력이 아니다
    stn = pd.read_csv("data/reference/stations_scored.csv")
    sat = causal_satellite(channels)

    df = sat.merge(lab, on=["STN", "date"], how="inner")
    df = df.merge(stn[["STN", "LAT", "LON", "HT"]], on="STN", how="inner")
    df["DOY"] = df["date"].dt.dayofyear
    df["DOY_SIN"] = np.sin(2 * np.pi * df["DOY"] / 365.25)
    df["DOY_COS"] = np.cos(2 * np.pi * df["DOY"] / 365.25)
    # 위성 천정각의 경로 길이 효과. 물리식에 sec(theta)-1 로 들어간다.
    from sme.core.grid import satellite_zenith
    _z = satellite_zenith(df["LON"].to_numpy(), df["LAT"].to_numpy())
    df["SEC_SATZEN"] = 1.0 / np.cos(np.radians(_z)) - 1.0
    df["STN_CAT"] = df["STN"].astype("category")
    # 연도. 규칙 2.1③ 이 'day, month, year' 를 허용 입력으로 명시한다.
    # 8월 말 기온이 2022년 24.8도 -> 2025년 31.4도로 빠르게 오르고 있어,
    # 연도 없이 과거를 넣으면 모델이 차갑게 편향된다(편향 -0.61). 선형
    # 성분에 연도를 주면 그 추세를 채점 연도로 이어 그릴 수 있다
    # (7년치 2.063 -> 1.875, 편향 -0.25).
    df["YEAR"] = df["date"].dt.year.astype(float)

    # 전국 대비 편차.
    # 각 행은 자기 지점 주변 9x9 만 본다. 그런데 기온은 그날 한반도를 덮은
    # 기단이 크게 좌우하므로, '오늘 전국이 얼마나 더운가' 를 모르면 지점
    # 특성과 그날 특성을 구분할 수 없다. 다른 지점의 '위성' 값을 쓰는 것이라
    # 규칙상 허용된다(ASOS 가 아니다).
    # 실전 재현 검증: 기온 RMSE 1.435 -> 1.304
    nat_keys = [c for c in df.columns if any(
        c.startswith(p_) for p_ in ("IR105_m9_", "IR105_min9_", "IR123_m9_",
                                    "SW038_m9_", "SWD_m9_", "SWDmin_"))]
    nat = df.groupby("date")[nat_keys].transform("mean")
    df = pd.concat([df, pd.DataFrame(
        {f"ANO_{c}": df[c] - nat[c] for c in nat_keys}, index=df.index)], axis=1)

    # 최근접 8개 지점의 위성 평균 (대상 시각만).
    # 전국 평균은 너무 넓고 3개는 너무 좁았다. 8개(대략 100~150km)가 기단
    # 규모와 맞는 듯하다 (기온 1.304 -> 1.268). 전 시각으로 넓히면 오히려
    # 나빠져(1.273) 대상 시각만 쓴다.
    nb_keys = [c for c in nat_keys if c.endswith("_h05") or c == "SWDmin_h05"]
    if nb_keys:
        la, lo = stn["LAT"].to_numpy(), stn["LON"].to_numpy()
        sid = stn["STN"].to_numpy()
        dist = np.sqrt(((la[:, None] - la) * 111) ** 2
                       + ((lo[:, None] - lo) * 111 * np.cos(np.radians(la[:, None]))) ** 2)
        np.fill_diagonal(dist, 1e9)
        near = {sid[i]: list(sid[np.argsort(dist[i])[:8]]) for i in range(len(sid))}
        for c in nb_keys:
            piv = df.pivot_table(index="date", columns="STN", values=c)
            m = pd.DataFrame({s_: piv[[x for x in near[s_] if x in piv.columns]].mean(axis=1)
                              for s_ in sid if s_ in piv.columns})
            st = m.stack().rename(f"NB_{c}").reset_index()
            st.columns = ["date", "STN", f"NB_{c}"]
            df = df.merge(st, on=["date", "STN"], how="left")

        # 방향별 이웃 (동/서/남/북 각 3개).
        # 8개 평균은 방향을 지운다. 서쪽 이웃이 뜨겁고 동쪽이 차가우면 서풍
        # 기단 유입이고, 동해안 푄도 '산맥 서쪽 구름 / 동쪽 맑음' 이라는
        # 방향 구조다. 실측: 기온 1.281 -> 1.243
        dy = (la[:, None] - la) * 111
        dx = (lo[:, None] - lo) * 111 * np.cos(np.radians(la[:, None]))
        sect = {"W": (dx < 0) & (np.abs(dx) > np.abs(dy)),
                "E": (dx > 0) & (np.abs(dx) > np.abs(dy)),
                "S": (dy < 0) & (np.abs(dy) >= np.abs(dx)),
                "N": (dy > 0) & (np.abs(dy) >= np.abs(dx))}
        for tag, mask in sect.items():
            grp = {}
            for i in range(len(sid)):
                cand = np.where(mask[i])[0]
                grp[sid[i]] = list(sid[cand[np.argsort(dist[i, cand])[:3]]]) if len(cand) else []
            for c in nb_keys:
                piv = df.pivot_table(index="date", columns="STN", values=c)
                m = pd.DataFrame({
                    s_: (piv[[x for x in grp[s_] if x in piv.columns]].mean(axis=1)
                         if grp.get(s_) else np.nan)
                    for s_ in sid if s_ in piv.columns})
                st = m.stack().rename(f"D{tag}_{c}").reset_index()
                st.columns = ["date", "STN", f"D{tag}_{c}"]
                df = df.merge(st, on=["date", "STN"], how="left")
        # 동서·남북 대비
        for c in nb_keys:
            if f"DE_{c}" in df and f"DW_{c}" in df:
                df[f"dEW_{c}"] = df[f"DE_{c}"] - df[f"DW_{c}"]
            if f"DN_{c}" in df and f"DS_{c}" in df:
                df[f"dNS_{c}"] = df[f"DS_{c}"] - df[f"DN_{c}"]
    # 평년값은 피처가 아니라 폴백용이지만, 비교 평가를 위해 붙여둔다.
    df = df.merge(climo, on=["STN", "DOY"], how="left")
    return df


# 모델 입력은 위성 + 정적 지점정보 + 달력뿐이다.
#
# 기후 평년값을 피처로 넣어봤지만 오히려 점수가 나빠졌다 (2.538 -> 2.577).
# 위경도·고도·연중일이 이미 같은 정보를 담고 있어 중복이기 때문이다.
# 빼는 편이 성능도 낫고, "ASOS 파생 자료를 입력으로 쓰는가"라는 규정
# 회색지대도 피할 수 있다. 평년값은 위성이 전부 결측일 때의 폴백으로만 쓴다.
#
# 해안거리·해양비율 같은 정적 지리 피처도 같은 이유로 제외했다
# (규정상 허용되지만 2.538 -> 2.545 로 도움이 되지 않았다).
BASE = ["LAT", "LON", "HT", "DOY_SIN", "DOY_COS", "SEC_SATZEN", "YEAR"]

# 지점번호. 규칙 2.1 이 station_list.csv 의 '지점번호' 를 허용 입력으로 명시한다.
# 위경도·고도만으로는 국지 현상(동해안 푄, 해풍 등)에서 생기는 지점별 계통
# 오차를 잡지 못한다. 범주형으로 넣으면 지점별 보정을 학습할 수 있다.
# GBM 에만 넣는다 — 선형 성분에 넣으면 의미 없는 크기 비교가 된다.
CAT = ["STN_CAT"]


def cv(df: pd.DataFrame, feats: list[str], target: str, n_splits: int = 4):
    X, y, g = df[feats], df[target], df["date"]
    oof = np.full(len(df), np.nan)
    for tr, va in GroupKFold(n_splits=n_splits).split(X, y, g):
        m = lgb.train(PARAMS, lgb.Dataset(X.iloc[tr], y.iloc[tr]),
                      num_boost_round=300)
        oof[va] = m.predict(X.iloc[va])
    ok = ~np.isnan(oof) & y.notna().to_numpy()
    return float(np.sqrt(((y.to_numpy()[ok] - oof[ok]) ** 2).mean()))


if __name__ == "__main__":
    CH = ["IR105", "IR123", "SW038"]
    df = assemble(CH)
    print(f"표본 {len(df):,}행 · {df['date'].nunique()}일 · "
          f"지점 {df['STN'].nunique()}개")
    print(f"사용 가능한 UTC 시각: 당일 ≤{TARGET_HOUR_UTC}시 + 전날\n")

    ta_c = float(np.sqrt(((df.TA14 - df.TA_CLIMO) ** 2).mean()))
    hm_c = float(np.sqrt(((df.HM14 - df.HM_CLIMO) ** 2).mean()))
    print(f"{'기후값 그대로':26s} RMSE_TA {ta_c:5.3f}  RMSE_HM {hm_c:6.3f}  "
          f"Score {score(ta_c, hm_c):6.3f}")

    ta_b, hm_b = cv(df, BASE, "TA14"), cv(df, BASE, "HM14")
    print(f"{'기후값+메타 (위성 없음)':26s} RMSE_TA {ta_b:5.3f}  "
          f"RMSE_HM {hm_b:6.3f}  Score {score(ta_b, hm_b):6.3f}")

    sf = BASE + [c for ch in CH for c in df.columns if c.startswith(ch + "_")]
    ta_s, hm_s = cv(df, sf, "TA14"), cv(df, sf, "HM14")
    print(f"{'+ 위성 3채널 (인과 준수)':26s} RMSE_TA {ta_s:5.3f}  "
          f"RMSE_HM {hm_s:6.3f}  Score {score(ta_s, hm_s):6.3f}")


## 4. 피처 조립

`evaluation.assemble()` 이 하는 일:

1. **창 통계** — 지점 주변 9x9(약 18km) 화소에서 중심값·평균·표준편차·최소·최대.
   중심 화소만 쓰면 구름 가장자리에서 값이 널뛴다. 표준편차 자체가 "구름이
   깨져 있는지"를 알려준다.
2. **시각별 전개** — 6개 관측 시각을 각각 별도 피처로 (`_h05`, `_p21` 등),
   그리고 시각 간 차분(`_d1h`, `_dnt`)으로 변화 속도를 담는다.
3. **분리대기창 보정항** — `SWD_* = IR105 - IR123`, `SWDmin_*`, 구름 지표 `CLD_*`.
4. **기하** — 위성 천정각의 `sec(theta)-1`, 연중일의 사인·코사인.
5. **전국 편차 `ANO_*`** — 그날 전국 평균 대비 이 지점의 편차. 타깃 분산의 66%가
   날짜 효과였으므로 이게 가장 큰 기여를 한다.
6. **이웃 `NB_*`** — 최근접 8개 지점의 평균.
7. **방향별 이웃 `DW_/DE_/DN_/DS_` 와 대비 `dEW_/dNS_`** — 8개 평균은 방향을
   지운다. 서쪽이 뜨겁고 동쪽이 차가우면 서풍 기단 유입이고, 동해안 푄도
   "산맥 서쪽 구름 / 동쪽 맑음"이라는 방향 구조다.
   실측: 기온 1.252 -> 1.238, 습도 7.309 -> 7.233.

In [ ]:
import numpy as np, pandas as pd, lightgbm as lgb, pickle
from sme.core import evaluation as E

CHANNELS = ["IR105", "IR123", "SW038"]
df = E.assemble(CHANNELS)

feats = E.BASE + [c for c in df.columns
                  if c.split("_")[0] in CHANNELS
                  or c.startswith(("SWD_", "SWDmin_", "CLD_"))]
feats = [f for f in dict.fromkeys(feats) if f in df.columns]
ano = [c for c in df.columns
       if c.startswith(("ANO_", "NB_", "DW_", "DE_", "DN_", "DS_", "dEW_", "dNS_"))]

print(f"표본 {len(df):,}행 · 날짜 {df['date'].nunique()}일 · 지점 {df['STN'].nunique()}개")
print(f"피처: 기본 {len(feats)}개 + 공간 {len(ano)}개")

## 5. 타깃마다 다른 피처를 쓴다

점수가 `RMSE_TA + 0.1*RMSE_HM` 이라 두 타깃을 각각 최적화하는 게 이득이다.
실측으로 갈린 두 가지:

- **연도(YEAR)**: 8월 말 기온은 연도 간 변동이 커서 넣어야 편향이 잡힌다
  (2026 검증 TA 1.506 -> 1.435, 편향 -0.54 -> +0.15). 그런데 습도는 연도 효과가
  계절과 뒤섞여 크게 나빠졌다 (HM 7.078 -> 9.654). 그래서 기온만 넣는다.
- **공간 피처**: 기온에는 크게 도움이 되지만 습도에서는 나빠졌다 (7.078 -> 7.340).

한 가지 시도해서 **기각한** 것도 남겨 둔다. 습도가 기온 잔차와 가장 강하게
연결(-0.476)되므로 *예측 습도*를 기온 입력으로 넣어봤다. 결과는 1.281 -> 1.283 로
효과가 없었다. 물리는 맞지만 우리 습도 예측 자체가 부정확해(RMSE 7%p) 그
신호가 노이즈에 묻힌 것이다.

In [ ]:
feats_by_target = {
    "TA14": feats + ano,
    "HM14": [f for f in feats if f != "YEAR"],   # 습도는 연도·공간 피처 제외
}
for t, f in feats_by_target.items():
    print(f"  {t}: {len(f)}개")

## 6. 선형 + GBM 잔차 구조

트리는 **학습 범위 밖으로 외삽하지 못한다**. 8월 자료로 학습한 GBM 단독 모델은
6월에도 "8월이니까 30도"를 그대로 내놓아 편향이 +4.3도였다. 그래서 Ridge 선형
모델을 앞에 두어 추세를 담당시키고, GBM 은 그 **잔차만** 학습한다. 계절 밖
편향이 사라졌을 뿐 아니라 계절 안 성능도 함께 좋아졌다.

지점번호(`STN_CAT`)는 GBM 에만 범주형으로 넣는다. 선형에 넣으면 번호의 크기를
비교하게 되는데 그건 아무 의미가 없다.

시드를 바꿔 학습한 모델 3개를 평균한다 (실측 2.282 -> 2.254). 시드가 고정되어
있고 `deterministic=True` 라서 다시 돌려도 같은 결과가 나온다 (규칙: 무작위성 제거).

In [ ]:
%%writefile sme/model/train.py
"""모델 학습 -> model.pkl.

산출물은 캐글 Dataset 으로 올려 추론 노트북에서 불러 쓴다.
추론 때 피처를 똑같이 만들어야 하므로, 피처 이름 순서와 기후값 표를
모델과 함께 한 묶음으로 저장한다.

재현성: 규정이 "무작위성 제거"를 요구한다. 시드를 고정하고 LightGBM 의
deterministic 옵션을 켠다.
"""
from __future__ import annotations

import pickle
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sme.core import evaluation
from sme.core import paths

SEED = 42
# 시드만 바꿔 학습한 모델을 평균낸다. 부스팅은 배깅·피처샘플링 때문에
# 시드마다 결과가 흔들리는데, 평균이 그 분산을 걷어낸다.
#   실측(홀드아웃 1주): 1개 2.282 -> 3개 2.254 -> 5개 2.250
# 시드가 고정되어 있으므로 재실행해도 같은 결과가 나온다 (규칙: 무작위성 제거).
SEEDS = [42, 7, 2024]
OUT = paths.MODEL_FILE

# 날짜 그룹 5-fold CV 로 고른 설정. 초기값(leaves31/lr.04/600r)은 표본이
# 59일이던 시절에 정한 것이라 데이터가 4배 늘어난 지금은 용량이 부족했다.
#   기준선 2.484 -> leaves63 2.467 -> +lr.02/1500r 2.458 -> leaves127 2.454
# 라운드는 1500 에서 포화했고(2500 도 동일), 선형 규제는 강할수록 나았다.
PARAMS = dict(objective="regression", metric="rmse", learning_rate=0.02,
              num_leaves=127, min_data_in_leaf=30, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              verbose=-1, seed=SEED, bagging_seed=SEED,
              feature_fraction_seed=SEED, deterministic=True,
              force_row_wise=True, num_threads=4)
ROUNDS = 1500
RIDGE_ALPHA = 500.0


def main() -> None:
    from sme.core import kma_client
    channels = list(kma_client.DEFAULT_CHANNELS)
    df = evaluation.assemble(channels)
    # 채널 파생 + 분리대기창 보정항(SWD_*) 을 모두 포함시킨다.
    feats = evaluation.BASE + [c for c in df.columns
                           if c.split("_")[0] in channels
                           or c.startswith(("SWD_", "SWDmin_", "CLD_"))]
    ano = [c for c in df.columns if c.startswith(("ANO_", "NB_", "DW_", "DE_", "DN_", "DS_", "dEW_", "dNS_"))]
    feats = [f for f in dict.fromkeys(feats) if f in df.columns]
    # 지점번호는 GBM 에만 넣는다 (선형에 넣으면 번호 크기를 비교하게 된다)
    gbm_feats = feats + evaluation.CAT

    # 연도는 타깃마다 다르게 쓴다.
    # 8월 말 기온은 연도 간 변동이 커서 연도를 넣어야 편향이 잡히지만
    # (2026 검증: TA 1.506 -> 1.435, 편향 -0.54 -> +0.15), 습도는 그 연도
    # 효과가 계절과 뒤섞여 오히려 크게 나빠졌다 (HM 7.078 -> 9.654).
    #   TA:연도O HM:연도X = 2026 검증 2.143 (최고), 2025 검증 1.862
    # 전국 편차와 이웃 평균은 기온에만 넣는다. 습도에서는 오히려 나빠졌다
    # (HM 7.078 -> 7.340). 지표가 TA + 0.1*HM 이라 각각 최적을 따로 쓴다.
    feats_by_target = {
        "TA14": feats + ano,
        "HM14": [f for f in feats if f != "YEAR"],
    }
    print(f"학습 표본 {len(df):,}행 · 피처 {len(feats)}개 · "
          f"날짜 {df['date'].nunique()}일")

    # 선형(Ridge) 을 앞에 두고 GBM 은 그 잔차만 학습한다.
    # 트리는 학습 범위 밖으로 외삽하지 못해 '8월이니까 30도' 를 6월에도
    # 그대로 내놓았다(편향 +4.3도). 선형항은 기울기를 따라 값이 이어지므로
    # 그 편향을 걷어낸다. 계절 안 성능도 함께 좋아졌다.
    models, linears = {}, {}
    for target in ("TA14", "HM14"):
        sub = df[df[target].notna()]
        tf = feats_by_target[target]
        tg = tf + evaluation.CAT
        X = sub[tf].astype(np.float64)
        y = sub[target].to_numpy(dtype=np.float64)

        lin = make_pipeline(SimpleImputer(strategy="median"),
                            StandardScaler(), Ridge(alpha=RIDGE_ALPHA))
        lin.fit(X, y)
        linears[target] = lin
        resid = y - lin.predict(X)

        ds = lgb.Dataset(sub[tg], resid, categorical_feature=evaluation.CAT)
        models[target] = [
            lgb.train(dict(PARAMS, seed=sd, bagging_seed=sd,
                           feature_fraction_seed=sd), ds, num_boost_round=ROUNDS)
            for sd in SEEDS
        ]
        print(f"  {target}: {len(sub):,}행 · 선형+GBM(시드 {len(SEEDS)}개) 학습 완료")

    # 기후 평년값은 싣지 않는다.
    # 운영진 답변: "ASOS는 모델 학습시 label값으로만 활용 가능하고
    # 그 외 입력값으로 활용하실 수 없습니다."
    # 지점별 과거 평균기온·기후 평년값을 추론에 쓰는 것은 금지 대상이다.
    # 위성이 결측이면 모델이 위경도·고도·연중일만으로 예측한다
    # (LightGBM 이 결측 피처를 스스로 처리한다).
    # 지점 좌표·고도는 대회가 제공한 station_list.csv 를 그대로 쓴다.
    # (운영진: "station_list 파일에 있는 정적인 정보를 사용하시면 됩니다")
    stn = pd.read_csv("data/reference/stations_scored.csv")

    # GK2A 는 정지위성이고 KO 격자는 고정이므로 지점별 화소 좌표가 상수다.
    # 미리 계산해 실어두면 추론 노트북에서 pyproj 가 필요 없어진다.
    from sme.core.grid import lonlat_to_rowcol, satellite_zenith
    grid = dict(origin_latitude=38.0, central_meridian=126.0,
                standard_parallel1=30.0, standard_parallel2=60.0,
                pixel_size=2000.0, upper_left_easting=-899000.0,
                upper_left_northing=899000.0)
    rows, cols = lonlat_to_rowcol(stn["LON"].to_numpy(),
                                  stn["LAT"].to_numpy(), grid)
    sat_zen = satellite_zenith(stn["LON"].to_numpy(), stn["LAT"].to_numpy())

    bundle = {
        "models": models,
        "linears": linears,   # 외삽 담당 선형 성분
        "features": feats,          # 전체 피처 (참고용)
        "gbm_features": gbm_feats,
        "features_by_target": feats_by_target,          # 선형 입력 (타깃별)
        "gbm_features_by_target": {k: v + evaluation.CAT
                                   for k, v in feats_by_target.items()},
        "categorical": evaluation.CAT,
        "stn_categories": list(df["STN_CAT"].cat.categories),
        "channels": channels,
        "win_stats": list(evaluation.WIN_STATS),
        "stations": stn,            # STN, LAT, LON, HT
        "sat_zenith": sat_zen,   # 지점별 위성 천정각(도) — 정지위성이라 상수
        "pixel_rows": rows,         # 지점별 화소 좌표 (2km 격자 고정)
        "pixel_cols": cols,
        "grid": grid,               # 검증용 격자 정의
        "target_hour_utc": evaluation.TARGET_HOUR_UTC,
        "hours_utc": [0, 2, 4, 5],          # 당일 (≤05 UTC = 14 KST)
        "hours_utc_prev": [18, 21],         # 전날 늦은 시각
        "seeds": SEEDS,
        "lightgbm_version": lgb.__version__,
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
    }
    with open(OUT, "wb") as f:
        pickle.dump(bundle, f, protocol=4)  # 구버전 호환 위해 protocol 4
    print(f"\n저장: {OUT}  ({OUT.stat().st_size/1e6:.2f} MB)")
    print("  기후 평년값 미포함 (ASOS 는 학습 라벨로만 사용)")
    print(f"  lightgbm {lgb.__version__} / pandas {pd.__version__} / "
          f"numpy {np.__version__}")
    print("  → 캐글 Dataset 으로 Public 업로드 후 노트북 Input 에 연결")


if __name__ == "__main__":
    main()


In [ ]:
from sme.model import train as T
print("설정:", {k: T.PARAMS[k] for k in
      ("learning_rate","num_leaves","min_data_in_leaf","feature_fraction","lambda_l2")})
print(f"라운드 {T.ROUNDS} · Ridge alpha {T.RIDGE_ALPHA} · 시드 {T.SEEDS}")

### 하이퍼파라미터를 고른 근거

날짜 그룹 5-fold CV 로 골랐다. 초기값(leaves 31 / lr 0.04 / 600라운드)은 표본이
59일이던 시절 것이라 데이터가 늘어난 뒤로는 용량이 부족했다.

    기준선 2.484 -> leaves 63 2.467 -> +lr 0.02/1500r 2.458 -> leaves 127 2.454

라운드는 1500에서 포화했고(2500도 동일), 선형 규제는 강할수록 나았다.

**검증은 반드시 날짜로 묶어야 한다.** 같은 날 96개 지점은 같은 기단을 공유하므로
무작위 분할은 사실상 정답을 흘린다. 홀드아웃 1주(672행)는 노이즈가 커서
0.03 정도 차이는 구별되지 않았고, 그래서 판단은 21,000행 이상의 5-fold CV 로 했다.

## 7. 학습 실행

`train.main()` 이 학습부터 `model.pkl` 저장까지 수행한다.

In [ ]:
T.main()

## 8. 규칙 준수

| 규칙 | 내용 | 준수 |
|---|---|---|
| 2.1 | 기상청 API 허브 자료만 사용 | GK2A LE1B + ASOS(라벨) 만 사용 |
| 2.2 | ASOS 는 학습 라벨로만 | 입력 피처에 ASOS 파생물 없음 |
| 2.3 | 외부 자료 금지 | 지점 좌표·고도는 대회 제공 `station_list.csv` |
| 3 | 14시 이하 관측만 | 05 UTC 이하 + 전날 18/21 UTC |
| 재현성 | 무작위성 제거 | 시드 고정 · `deterministic=True` |

**기후 평년값은 싣지 않았다.** 운영진 답변("ASOS는 모델 학습시 label값으로만
활용 가능")에 따라 지점별 과거 평균기온을 추론에 쓰는 것은 금지 대상으로 보고
번들에서 제외했다. 위성이 결측이면 모델이 위경도·고도·연중일만으로 예측한다.

## 9. 성능

실제 채점과 같은 조건 — 학습은 검증 주 **이전 날짜만**, 미래를 전혀 보지 않는다.

검증 주간 2026-08-16~22 (668행):

| 구성 | 점수 | 기온 | 습도 |
|---|---:|---:|---:|
| 7년치 + 연도 피처 | 2.401 | | |
| 타깃별 연도 분리 | 2.130 | | |
| + 전국 편차 | 2.009 | | |
| + 최근접 8개 이웃 | 1.983 | 1.252 | 7.309 |
| **+ 방향별 이웃 (최종)** | **1.961** | **1.238** | **7.233** |

### 정보의 한계

같은 주간에 대해 "가능한 최선"을 직접 재봤다.

| 방법 | 점수 |
|---|---:|
| 14시 ASOS 그대로 베끼기 | 0.000 |
| 13시 ASOS 그대로 베끼기 | 1.345 |
| 다른 95개 지점의 실제 14시 값으로 완벽 공간보간 | 2.173 |
| **우리 모델 (위성만)** | **1.961** |

세 번째 줄이 중요하다. **다른 모든 지점의 정답을 알아도** 공간보간만으로는
2.173 이다. 위성만 쓰는 우리 모델이 그보다 낫다는 것은, 위성이 지점별
국지 상태에 대해 이웃 관측에 없는 정보를 실제로 담고 있다는 뜻이다.